In [1]:
import json
import os
import tkinter as tk
from tkinter import ttk, messagebox
from datetime import datetime


class Book:
    """Класс для хранения информации о книге"""
    
    def __init__(self, title, author, genre, pages, date_added=None):
        self.title = title.strip()
        self.author = author.strip()
        self.genre = genre.strip()
        self.pages = int(pages)
        self.date_added = date_added or datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    def to_dict(self):
        """Преобразует книгу в словарь для JSON"""
        return {
            'title': self.title,
            'author': self.author,
            'genre': self.genre,
            'pages': self.pages,
            'date_added': self.date_added
        }
    
    @classmethod
    def from_dict(cls, data):
        """Создаёт книгу из словаря"""
        return cls(
            data['title'],
            data['author'],
            data['genre'],
            data['pages'],
            data.get('date_added')
        )


class BookTrackerApp:
    """GUI-приложение для отслеживания книг"""
    
    def __init__(self):
        self.root = tk.Tk()
        self.root.title("Book Tracker - Трекер прочитанных книг")
        self.root.geometry("1000x700")
        self.root.resizable(True, True)
        
        self.books = []
        self.filtered_books = []
        
        # Загружаем данные
        self.load_books()
        
        # Создаём интерфейс
        self.setup_ui()
        
        # Обновляем таблицу
        self.refresh_table()
    
    def setup_ui(self):
        """Создание интерфейса пользователя"""
        
        # Основная рамка
        main_frame = ttk.Frame(self.root, padding="10")
        main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Заголовок
        title_label = ttk.Label(main_frame, text="Book Tracker", font=("Arial", 20, "bold"))
        title_label.pack(pady=10)
        
        # Рамка для добавления книги
        add_frame = ttk.LabelFrame(main_frame, text="Добавить новую книгу", padding="10")
        add_frame.pack(fill=tk.X, pady=10)
        
        # Поля ввода
        # Название
        ttk.Label(add_frame, text="Название книги:").grid(row=0, column=0, sticky=tk.W, pady=5, padx=5)
        self.title_entry = ttk.Entry(add_frame, width=40)
        self.title_entry.grid(row=0, column=1, pady=5, padx=5, sticky=tk.W)
        
        # Автор
        ttk.Label(add_frame, text="Автор:").grid(row=1, column=0, sticky=tk.W, pady=5, padx=5)
        self.author_entry = ttk.Entry(add_frame, width=40)
        self.author_entry.grid(row=1, column=1, pady=5, padx=5, sticky=tk.W)
        
        # Жанр
        ttk.Label(add_frame, text="Жанр:").grid(row=2, column=0, sticky=tk.W, pady=5, padx=5)
        self.genre_entry = ttk.Entry(add_frame, width=40)
        self.genre_entry.grid(row=2, column=1, pady=5, padx=5, sticky=tk.W)
        
        # Количество страниц
        ttk.Label(add_frame, text="Количество страниц:").grid(row=3, column=0, sticky=tk.W, pady=5, padx=5)
        self.pages_entry = ttk.Entry(add_frame, width=40)
        self.pages_entry.grid(row=3, column=1, pady=5, padx=5, sticky=tk.W)
        
        # Кнопка добавления
        add_btn = ttk.Button(add_frame, text="Добавить книгу", command=self.add_book)
        add_btn.grid(row=4, column=1, pady=10, sticky=tk.W, padx=5)
        
        # Рамка для фильтрации
        filter_frame = ttk.LabelFrame(main_frame, text="Фильтрация", padding="10")
        filter_frame.pack(fill=tk.X, pady=10)
        
        # Фильтр по жанру
        ttk.Label(filter_frame, text="Фильтр по жанру:").grid(row=0, column=0, sticky=tk.W, pady=5, padx=5)
        self.genre_filter = ttk.Entry(filter_frame, width=30)
        self.genre_filter.grid(row=0, column=1, pady=5, padx=5, sticky=tk.W)
        self.genre_filter.bind('<KeyRelease>', self.apply_filters)
        
        # Фильтр по количеству страниц
        ttk.Label(filter_frame, text="Страниц больше чем:").grid(row=1, column=0, sticky=tk.W, pady=5, padx=5)
        self.pages_filter = ttk.Entry(filter_frame, width=30)
        self.pages_filter.grid(row=1, column=1, pady=5, padx=5, sticky=tk.W)
        self.pages_filter.bind('<KeyRelease>', self.apply_filters)
        
        # Кнопки фильтрации
        filter_btn_frame = ttk.Frame(filter_frame)
        filter_btn_frame.grid(row=2, column=1, pady=10, sticky=tk.W)
        
        clear_filters_btn = ttk.Button(filter_btn_frame, text="Очистить фильтры", command=self.clear_filters)
        clear_filters_btn.pack(side=tk.LEFT, padx=5)
        
        # Рамка для статистики
        stats_frame = ttk.LabelFrame(main_frame, text="Статистика", padding="10")
        stats_frame.pack(fill=tk.X, pady=10)
        
        self.stats_label = ttk.Label(stats_frame, text="", font=("Arial", 10))
        self.stats_label.pack()
        
        # Рамка для таблицы книг
        table_frame = ttk.LabelFrame(main_frame, text="Список книг", padding="10")
        table_frame.pack(fill=tk.BOTH, expand=True, pady=10)
        
        # Создаём таблицу
        columns = ('Название', 'Автор', 'Жанр', 'Страниц', 'Дата добавления')
        self.tree = ttk.Treeview(table_frame, columns=columns, show='headings', height=15)
        
        # Настройка заголовков
        for col in columns:
            self.tree.heading(col, text=col)
            self.tree.column(col, width=150)
        
        # Настройка ширины колонок
        self.tree.column('Название', width=250)
        self.tree.column('Автор', width=150)
        self.tree.column('Жанр', width=120)
        self.tree.column('Страниц', width=80)
        self.tree.column('Дата добавления', width=150)
        
        # Добавляем скроллбар
        scrollbar = ttk.Scrollbar(table_frame, orient=tk.VERTICAL, command=self.tree.yview)
        self.tree.configure(yscrollcommand=scrollbar.set)
        
        self.tree.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        
        # Контекстное меню для удаления
        self.context_menu = tk.Menu(self.root, tearoff=0)
        self.context_menu.add_command(label="Удалить книгу", command=self.delete_book)
        self.tree.bind("<Button-3>", self.show_context_menu)
        
        # Кнопка удаления
        delete_btn = ttk.Button(main_frame, text="Удалить выбранную книгу", command=self.delete_book)
        delete_btn.pack(pady=5)
        
        # Статусная строка
        self.status_var = tk.StringVar()
        self.status_var.set("Готов к работе")
        status_bar = ttk.Label(main_frame, textvariable=self.status_var, relief=tk.SUNKEN)
        status_bar.pack(fill=tk.X, pady=5)
    
    def add_book(self):
        """Добавляет новую книгу с валидацией"""
        
        # Получаем данные из полей
        title = self.title_entry.get().strip()
        author = self.author_entry.get().strip()
        genre = self.genre_entry.get().strip()
        pages_str = self.pages_entry.get().strip()
        
        # Валидация: проверка на пустые поля
        if not title:
            messagebox.showerror("Ошибка", "Название книги не может быть пустым")
            return
        
        if not author:
            messagebox.showerror("Ошибка", "Автор не может быть пустым")
            return
        
        if not genre:
            messagebox.showerror("Ошибка", "Жанр не может быть пустым")
            return
        
        if not pages_str:
            messagebox.showerror("Ошибка", "Количество страниц не может быть пустым")
            return
        
        # Валидация: количество страниц должно быть числом
        try:
            pages = int(pages_str)
            if pages <= 0:
                messagebox.showerror("Ошибка", "Количество страниц должно быть положительным числом")
                return
        except ValueError:
            messagebox.showerror("Ошибка", "Количество страниц должно быть числом")
            return
        
        # Создаём книгу
        book = Book(title, author, genre, pages)
        self.books.append(book)
        
        # Сохраняем в JSON
        if self.save_books():
            # Очищаем поля
            self.title_entry.delete(0, tk.END)
            self.author_entry.delete(0, tk.END)
            self.genre_entry.delete(0, tk.END)
            self.pages_entry.delete(0, tk.END)
            
            # Обновляем отображение
            self.refresh_table()
            self.update_statistics()
            
            self.status_var.set(f"Добавлена книга: {title} - {author}")
            messagebox.showinfo("Успех", f"Книга '{title}' успешно добавлена!")
    
    def delete_book(self):
        """Удаляет выбранную книгу"""
        selection = self.tree.selection()
        if not selection:
            messagebox.showwarning("Нет выбора", "Выберите книгу для удаления")
            return
        
        # Получаем индекс выбранной книги
        item = self.tree.item(selection[0])
        values = item['values']
        
        # Находим книгу в списке
        for i, book in enumerate(self.filtered_books):
            if (book.title == values[0] and book.author == values[1] and 
                book.genre == values[2] and book.pages == int(values[3])):
                
            if (book.title == values[0] and book.author == values[1] and 
                book.genre == values[2] and book.pages == int(values[3])):
                if messagebox.askyesno("Подтверждение", f"Удалить книгу '{book.title}'?"):
                    self.books.remove(book)
                    self.save_books()
                    self.refresh_table()
                    self.update_statistics()
                    self.status_var.set(f"Удалена книга: {book.title}")
                break
    
    def show_context_menu(self, event):
        """Показывает контекстное меню"""
        item = self.tree.identify_row(event.y)
        if item:
            self.tree.selection_set(item)
            self.context_menu.post(event.x_root, event.y_root)
    
    def apply_filters(self, event=None):
        """Применяет фильтры к списку книг"""
        genre_filter = self.genre_filter.get().strip().lower()
        pages_filter_str = self.pages_filter.get().strip()
        
        self.filtered_books = self.books.copy()
        
        # Фильтр по жанру
        if genre_filter:
            self.filtered_books = [b for b in self.filtered_books if genre_filter in b.genre.lower()]
        
        # Фильтр по страницам (больше указанного числа)
        if pages_filter_str:
            try:
                pages_min = int(pages_filter_str)
                self.filtered_books = [b for b in self.filtered_books if b.pages > pages_min]
            except ValueError:
                pass
        
        self.refresh_table(filtered=True)
        self.status_var.set(f"Найдено книг: {len(self.filtered_books)}")
    
    def clear_filters(self):
        """Очищает все фильтры"""
        self.genre_filter.delete(0, tk.END)
        self.pages_filter.delete(0, tk.END)
        self.filtered_books = self.books.copy()
        self.refresh_table(filtered=True)
        self.status_var.set(f"Фильтры очищены. Всего книг: {len(self.books)}")
    
    def refresh_table(self, filtered=False):
        """Обновляет таблицу книг"""
        # Очищаем таблицу
        for item in self.tree.get_children():
            self.tree.delete(item)
        
        # Выбираем источник данных
        books_to_show = self.filtered_books if filtered else self.books
        
        # Добавляем книги в таблицу
        for book in books_to_show:
            self.tree.insert('', tk.END, values=(
                book.title,
                book.author,
                book.genre,
                book.pages,
                book.date_added
            ))
    
    def update_statistics(self):
        """Обновляет статистику"""
        if not self.books:
            self.stats_label.config(text="Нет добавленных книг")
            return
        
        total_books = len(self.books)
        total_pages = sum(book.pages for book in self.books)
        avg_pages = total_pages // total_books if total_books > 0 else 0
        
        # Уникальные жанры и авторы
        unique_genres = len(set(book.genre for book in self.books))
        unique_authors = len(set(book.author for book in self.books))
        
        stats_text = f"Всего книг: {total_books} | Всего страниц: {total_pages} | "
        stats_text += f"Среднее страниц: {avg_pages} | Жанров: {unique_genres} | Авторов: {unique_authors}"
        
        self.stats_label.config(text=stats_text)
    
    def load_books(self):
        """Загружает книги из JSON файла"""
        if os.path.exists('books.json'):
            try:
                with open('books.json', 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    self.books = [Book.from_dict(item) for item in data]
                print(f"Загружено {len(self.books)} книг")
                self.status_var.set(f"Загружено {len(self.books)} книг")
            except Exception as e:
                print(f"Ошибка загрузки: {e}")
                self.books = []
        else:
            self.books = []
            print("Файл books.json не найден. Будет создан новый.")
    
    def save_books(self):
        """Сохраняет книги в JSON файл"""
        try:
            with open('books.json', 'w', encoding='utf-8') as f:
                json.dump([book.to_dict() for book in self.books], f, ensure_ascii=False, indent=2)
            return True
        except Exception as e:
            messagebox.showerror("Ошибка", f"Не удалось сохранить данные: {e}")
            return False
    
    def run(self):
        """Запускает приложение"""
        self.update_statistics()
        self.root.mainloop()


def main():
    """Точка входа в программу"""
    app = BookTrackerApp()
    app.run()


if __name__ == "__main__":
    main()

IndentationError: expected an indented block after 'if' statement on line 241 (1323266259.py, line 244)